## In this notebook:
#### We import PET-derived volumetric receptor maps and parcellate using the Schaefer100 atlas.

1. Import packages and maps
2. Parcellate


### Import Packages

In [1]:
import pandas as pd
from nilearn.datasets import fetch_atlas_schaefer_2018
from neuromaps.parcellate import Parcellater
import warnings
from neuromaps.datasets import fetch_fslr, fetch_annotation
from neuromaps import datasets, images, nulls, resampling, stats, transforms, parcellate
from neuromaps.parcellate import Parcellater 
from neuromaps.resampling import resample_images
from neuromaps.images import dlabel_to_gifti
import neuromaps
import nibabel as nib
import os
import numpy as np
import mayavi
from PIL import Image
import hcp_utils as hcp
import nilearn.plotting as plotting
import warnings
from netneurotools.datasets import fetch_cammoun2012
#import abagen
#import abagen ## This is important for certification purposes 
warnings.filterwarnings('ignore')

pixdim[1,2,3] should be non-zero; setting 0 dims to 1


### Set Variables

In [2]:
#schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
#cammoun = fetch_cammoun2012()
desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikan-killiany2006_rois_1mm.nii.gz'
desikan = '/Users/pecsok/nilearn_data/desikan_killiany/desikanKillianyMNI.nii.gz'
atlas = 'desikan'

#scale = 'scale500'
scale = 'scale1000_17'
path = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/"
outpath = "/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/parcellated/PET_parcellated/desikan_killiany/"

#print(schaefer['maps'])
print(desikan)



/Users/pecsok/nilearn_data/desikan_killiany/desikanKillianyMNI.nii.gz


### Load in maps of interest

In [3]:
receptors_nii = [path+'5HT1a_way_hc36_savli.nii',
                 path+'5HT1a_cumi_hc8_beliveau.nii',
                 path+'5HT1b_az_hc36_beliveau.nii',
                 path+'5HT1b_p943_hc22_savli.nii',
                 path+'5HT1b_p943_hc65_gallezot.nii.gz',
                 path+'5HT2a_cimbi_hc29_beliveau.nii',
                 path+'5HT2a_alt_hc19_savli.nii',
                 path+'5HT2a_mdl_hc3_talbot.nii.gz',
                 path+'5HT4_sb20_hc59_beliveau.nii',
                 path+'5HT6_gsk_hc30_radhakrishnan.nii.gz',
                 path+'5HTT_dasb_hc100_beliveau.nii',
                 path+'5HTT_dasb_hc30_savli.nii',
                 path+'A4B2_flubatine_hc30_hillmer.nii.gz',
                 path+'CB1_omar_hc77_normandin.nii.gz',
                 path+'CB1_FMPEPd2_hc22_laurikainen.nii',
                 path+'D1_SCH23390_hc13_kaller.nii',
                 path+'D2_fallypride_hc49_jaworska.nii',
                 path+'D2_flb457_hc37_smith.nii.gz',
                 path+'D2_flb457_hc55_sandiego.nii.gz',
                 path+'D2_raclopride_hc7_alakurtti.nii',
                 path+'DAT_fpcit_hc174_dukart_spect.nii',
                 path+'DAT_fepe2i_hc6_sasaki.nii.gz',
                 #path+'GABAa-bz_flumazenil_hc16_norgaard.nii',
                 path+'GABAa_flumazenil_hc6_dukart.nii',
                 path+'H3_cban_hc8_gallezot.nii.gz',
                 path+'M1_lsn_hc24_naganawa.nii.gz',
                 path+'mGluR5_abp_hc22_rosaneto.nii',
                 path+'mGluR5_abp_hc28_dubois.nii',
                 path+'mGluR5_abp_hc73_smart.nii',
                 path+'MU_carfentanil_hc204_kantonen.nii',
                 path+'MU_carfentanil_hc39_turtonen.nii',
                 path+'NAT_MRB_hc77_ding.nii.gz',
                 path+'NAT_MRB_hc10_hesse.nii',
                 path+'NMDA_ge179_hc29_galovic.nii.gz',
                 path+'VAChT_feobv_hc4_tuominen.nii',
                 path+'VAChT_feobv_hc5_bedard_sum.nii',
                 path+'VAChT_feobv_hc18_aghourian_sum.nii']

"""
[path+'GABAa-bz_flumazenil_hc16_norgaard.nii',
                 path+'GABAa_flumazenil_hc6_dukart.nii',
                 path+'mGluR5_abp_hc22_rosaneto.nii',
                 path+'mGluR5_abp_hc28_dubois.nii',
                 path+'mGluR5_abp_hc73_smart.nii',
                 path+'NMDA_ge179_hc29_galovic.nii.gz',
                 path+'DAT_fpcit_hc174_dukart_spect.nii',
                 path+'DAT_fepe2i_hc6_sasaki.nii.gz',
                 path+'CB1_omar_hc77_normandin.nii.gz',
                 path+'CB1_FMPEPd2_hc22_laurikainen.nii'
                 path+'VAChT_feobv_hc4_tuominen.nii',
                 path+'VAChT_feobv_hc5_bedard_sum.nii',
                 path+'VAChT_feobv_hc18_aghourian_sum.nii']
"""

"\n[path+'GABAa-bz_flumazenil_hc16_norgaard.nii',\n                 path+'GABAa_flumazenil_hc6_dukart.nii',\n                 path+'mGluR5_abp_hc22_rosaneto.nii',\n                 path+'mGluR5_abp_hc28_dubois.nii',\n                 path+'mGluR5_abp_hc73_smart.nii',\n                 path+'NMDA_ge179_hc29_galovic.nii.gz',\n                 path+'DAT_fpcit_hc174_dukart_spect.nii',\n                 path+'DAT_fepe2i_hc6_sasaki.nii.gz',\n                 path+'CB1_omar_hc77_normandin.nii.gz',\n                 path+'CB1_FMPEPd2_hc22_laurikainen.nii'\n                 path+'VAChT_feobv_hc4_tuominen.nii',\n                 path+'VAChT_feobv_hc5_bedard_sum.nii',\n                 path+'VAChT_feobv_hc18_aghourian_sum.nii']\n"

### Parcellate

In [4]:
if atlas == 'schaefer': 
    parcellated = {}
    parcellater = Parcellater(schaefer['maps'], 'MNI152')
    for receptor in receptors_nii:
        parcellated[receptor] = parcellater.fit_transform(receptor, 'MNI152', True)
        #print(parcellated[receptor])
        print(parcellated[receptor].shape)
        name = receptor.split('/')[-1]  # get nifti file name
        name = name.split('.')[0]  # remove .nii
        np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')
elif atlas == 'cammoun':
    parcellated = {}
    parcellater = Parcellater(cammoun[scale], 'MNI152')
    for receptor in receptors_nii:
        parcellated[receptor] = parcellater.fit_transform(receptor, 'MNI152', True)
        print(parcellated[receptor])
        print(parcellated[receptor].shape)
        name = receptor.split('/')[-1]  # get nifti file name
        name = name.split('.')[0]  # remove .nii
        np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')

elif atlas == 'desikan':
    parcellated = {}
    parcellater = Parcellater(desikan, 'MNI152')
    for receptor in receptors_nii:
        parcellated[receptor] = parcellater.fit_transform(receptor, 'MNI152', True)
        #print(parcellated[receptor])
        print(parcellated[receptor].shape)
        name = receptor.split('/')[-1]  # get nifti file name
        name = name.split('.')[0]  # remove .nii
        np.savetxt(outpath+name+'.csv', parcellated[receptor], delimiter=',')

(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)
(1, 112)


In [5]:
#print(parcellated['/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/GABAa-bz_flumazenil_hc16_norgaard.nii'])

print(parcellated['/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/GABAa_flumazenil_hc6_dukart.nii'])



#print(parcellated)

KeyError: '/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/maps/PET_nifti_images/GABAa-bz_flumazenil_hc16_norgaard.nii'

In [ ]:
# Desikan Killiany Atlas
